# BREATH API – Python Demo

**BREATH** (Biophysical Rhythm of Ecosystem Activity & Health) simulates gross primary productivity (GPP), ecosystem respiration (RECO), and net ecosystem exchange (NEE) at hourly resolution using the VPRM photosynthesis model and SWELL phenology.

This notebook shows how to:
1. Submit a simulation via `POST /api/breath/run`
2. Poll `GET /api/breath/status` until the run finishes
3. Download results from `GET /api/results/latest`
4. Compute annual carbon budgets and plot GPP, RECO, NEE

**Requirements**
```
pip install requests pandas matplotlib
```

Set `BASE_URL` to your deployed instance (Render) or `http://localhost:5244` for local dev.

In [ ]:
import time
import io
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

BASE_URL = "https://breath-api-thkm.onrender.com"  # change to http://localhost:5244 for local

## 1 · Submit a simulation

A pixel is identified by `"lat_lon"` (4 decimal places). Here we simulate a deciduous broadleaf forest in the Po Valley, Italy.

In [ ]:
payload = {
    "settings": {
        "pixelsRun":           ["44.6813_11.0217"],   # lat_lon of the forest pixel
        "startYear":           2022,
        "endYear":             2024,
        "inputWeather":        "hourly",               # "hourly" (NASA POWER) or "daily"
        "modelVariant":        "Circadian",            # "Baseline" | "Pheno" | "Circadian"
        "calibration":         False,                  # True = calibrate against MODIS EVI
        "calibrationVariable": "Phenology",
        "simplexes":           3,
        "iterations":          200,
        "parametersDataFile":  "photothermalRequirements.csv",
    }
}

resp = requests.post(f"{BASE_URL}/api/breath/run", json=payload, timeout=30)
resp.raise_for_status()
print(resp.json())

## 2 · Poll until complete

`GET /api/breath/status` returns `{"Status": "Running" | "Completed" | "Failed", ...}`

In [ ]:
def wait_for_completion(base_url: str, poll_interval: int = 10, timeout: int = 600) -> dict:
    deadline = time.time() + timeout
    while time.time() < deadline:
        status = requests.get(f"{base_url}/api/breath/status", timeout=10).json()
        state  = status.get("Status", "")
        print(f"  {state} …", end="\r")
        if state == "Completed":
            print(f"\nDone in {status.get('DurationMs', '?')} ms")
            return status
        if state == "Failed":
            raise RuntimeError("Simulation failed — check /api/breath/stream/logs")
        time.sleep(poll_interval)
    raise TimeoutError(f"Simulation did not finish within {timeout}s")

run_info = wait_for_completion(BASE_URL)

## 3 · Download and parse results

`GET /api/results/latest` returns a CSV with one row per simulated hour.

In [ ]:
csv_text = requests.get(f"{BASE_URL}/api/results/latest", timeout=30).text
df = pd.read_csv(io.StringIO(csv_text), parse_dates=["date"])

print(f"Rows: {len(df):,}  |  Columns: {df.columns.tolist()}")
df.head(3)

## 4 · Annual carbon budget

Daily mean fluxes (µmol CO₂ m⁻² s⁻¹) × 86 400 s/day × 12.01 g/mol × 10⁻⁶ mol/µmol = gC m⁻² day⁻¹

In [ ]:
UMOL_TO_GC_DAY = 86_400 * 12.01e-6   # µmol m⁻² s⁻¹ → gC m⁻² day⁻¹

daily = df.groupby("date")[["GPP", "RECO", "NEE"]].mean() * UMOL_TO_GC_DAY

annual = daily.resample("YE").sum().rename_axis("Year")
annual.index = annual.index.year
print(annual.round(1).to_string())

## 5 · Time-series plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
fig.suptitle("BREATH simulation – Po Valley forest pixel", fontsize=13)

# SWELL phenology index
if "SWELL" in df.columns:
    swell_d = df.groupby("date")["SWELL"].mean()
    ax1.fill_between(swell_d.index, swell_d, alpha=0.35, color="#facc15", label="SWELL (EVI)")
    ax1.plot(swell_d.index, swell_d, color="#ca8a04", linewidth=0.8)
    ax1.set_ylabel("EVI (0–1)", fontsize=9)
    ax1.legend(fontsize=9, loc="upper right")
    ax1.set_ylim(0, 1)

# Carbon fluxes
ax2.plot(daily.index, daily["GPP"],  color="#4ade80", linewidth=1, label="GPP")
ax2.plot(daily.index, daily["RECO"], color="#f87171", linewidth=1, label="RECO")
ax2.axhline(0, color="#475569", linewidth=0.6, linestyle="--")
ax2.fill_between(daily.index, daily["NEE"], 0,
                 where=(daily["NEE"] < 0), alpha=0.25, color="#60a5fa", label="NEE (sink)")
ax2.fill_between(daily.index, daily["NEE"], 0,
                 where=(daily["NEE"] >= 0), alpha=0.25, color="#f97316", label="NEE (source)")
ax2.set_ylabel("gC m⁻² day⁻¹", fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.tick_params(axis="x", rotation=30, labelsize=8)
ax2.legend(fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig("breath_fluxes.png", dpi=150)
plt.show()

## 6 · Multi-pixel grid run

Pass multiple `"lat_lon"` strings in `pixelsRun` to run a spatial grid (max 25 pixels per run).

In [ ]:
import itertools

# 3 × 3 grid centred on the Po Valley pixel, 0.1° step
lats = [44.58, 44.68, 44.78]
lons = [10.92, 11.02, 11.12]
pixels = [f"{lat:.4f}_{lon:.4f}" for lat, lon in itertools.product(lats, lons)]

grid_payload = {
    "settings": {
        "pixelsRun":          pixels,
        "startYear":          2023,
        "endYear":            2023,
        "inputWeather":       "hourly",
        "modelVariant":       "Circadian",
        "calibration":        False,
        "parametersDataFile": "photothermalRequirements.csv",
        "simplexes":          3,
        "iterations":         200,
        "calibrationVariable": "Phenology",
    }
}

requests.post(f"{BASE_URL}/api/breath/run", json=grid_payload).raise_for_status()
run_info = wait_for_completion(BASE_URL)

In [ ]:
csv_text = requests.get(f"{BASE_URL}/api/results/latest", timeout=30).text
df_grid  = pd.read_csv(io.StringIO(csv_text))

# Annual NEE per pixel
UMOL_TO_GC_YEAR = 86_400 * 365 * 12.01e-6
nee_px = (
    df_grid.groupby("pixel")["NEE"]
    .mean()
    .mul(UMOL_TO_GC_YEAR)
    .round(1)
    .rename("Annual NEE (gC m⁻² yr⁻¹)")
)
print(nee_px.sort_values().to_string())

## 7 · Custom parameter overrides

Any model parameter can be overridden by adding a `parameterOverrides` dict. Keys must match the `className_propertyName` format used internally.

In [ ]:
tuned_payload = {
    "settings": {
        "pixelsRun":          ["44.6813_11.0217"],
        "startYear":          2023,
        "endYear":            2023,
        "inputWeather":       "hourly",
        "modelVariant":       "Circadian",
        "calibration":        False,
        "parametersDataFile": "photothermalRequirements.csv",
        "simplexes":          3,
        "iterations":         200,
        "calibrationVariable": "Phenology",
    },
    "parameterOverrides": {
        "parPhotosynthesis_maximumQuantumYieldOver": 0.08,   # α overstory (default 0.06)
        "parPhotosynthesis_halfSaturationTree":      350,    # k½ PAR (default 400)
        "parRespiration_referenceRespiration":        1.2,   # R base (default 1.5)
    }
}

requests.post(f"{BASE_URL}/api/breath/run", json=tuned_payload).raise_for_status()
wait_for_completion(BASE_URL)
print("Done — retrieve results as usual.")

## 8 · Custom weather data

By default BREATH downloads meteorological inputs automatically from NASA POWER.
If you have your own station data, reanalysis output, or ERA5 downloads you can
upload a daily weather CSV and the next simulation for that pixel will use it
instead of the remote download.

### Required columns

| Column | Unit |
|--------|------|
| `Date` | yyyy-MM-dd |
| `PAR` | MJ m⁻² d⁻¹ |
| `airTemperatureMaximum` | °C |
| `airTemperatureMinimum` | °C |
| `solarRadiation` | MJ m⁻² d⁻¹ |
| `dewPointTemperature` | °C |
| `precipitation` | mm d⁻¹ |
| `latitude` | ° |


In [ ]:
# Fetch the server-side template to see exact column format
template = requests.get(f"{BASE_URL}/api/weather/template", timeout=10).text
print(template)


In [ ]:
import io
import pandas as pd

PIXEL_CUSTOM = "44.6813_11.0217"
LAT_CUSTOM   = 44.6813

# Build a synthetic daily weather file (replace with real observations)
dates = pd.date_range("2022-01-01", "2024-12-31", freq="D")
weather_df = pd.DataFrame({
    "Date":                   dates.strftime("%Y-%m-%d"),
    "PAR":                    5.0,
    "airTemperatureMaximum":  12.0,
    "airTemperatureMinimum":   2.0,
    "solarRadiation":         10.0,
    "dewPointTemperature":    -1.0,
    "precipitation":           1.5,
    "latitude":               LAT_CUSTOM,
})

# Upload — stored server-side as {pixel}_weather.csv
csv_bytes = weather_df.to_csv(index=False).encode()
resp_upload = requests.post(
    f"{BASE_URL}/api/weather/upload",
    params={"pixel": PIXEL_CUSTOM},
    data=csv_bytes,
    headers={"Content-Type": "text/csv"},
    timeout=30,
)
print(resp_upload.json())


In [ ]:
# Run with custom weather — BREATH detects the cached file automatically
custom_payload = {
    "settings": {
        "pixelsRun":           [PIXEL_CUSTOM],
        "startYear":           2022,
        "endYear":             2024,
        "inputWeather":        "daily",   # must match the file resolution
        "modelVariant":        "Pheno",
        "calibration":         False,
        "parametersDataFile":  "photothermalRequirements.csv",
        "simplexes":           3,
        "iterations":          200,
        "calibrationVariable": "Phenology",
    }
}

requests.post(f"{BASE_URL}/api/breath/run", json=custom_payload, timeout=30).json()


In [ ]:
wait_for_completion(BASE_URL)

# Download and inspect results
df_custom = pd.read_csv(io.StringIO(
    requests.get(f"{BASE_URL}/api/results/latest", timeout=30).text
))
print(df_custom[["date", "GPP", "RECO", "NEE"]].head())


In [ ]:
# List all pixels with a cached weather file
print(requests.get(f"{BASE_URL}/api/weather/list", timeout=10).json())

# Delete a specific pixel weather file (reverts to NASA POWER on next run)
requests.delete(f"{BASE_URL}/api/weather/{PIXEL_CUSTOM}", timeout=10).json()
